# Phase 3b — MMDetection Route B (native `mmdet` + RTMDet)

Rebuilds the detection stage on native **MMDetection** (RTMDet-s) instead of Ultralytics,
per the supervisor's ask, and integrates the enhancement/defogging transform as the
thesis's contribution.

Full background: `docs/MMDETECTION_ROUTE_B_RUNBOOK.md` (local-only, not pushed — see repo README).

**Run order:** mount Drive -> clone/pull repo -> §0.5 (download + preprocess BDD100K) ->
§Env setup (Cells 1-7, run once per session, does NOT survive a Colab disconnect) ->
Task 2 (YOLO->COCO) -> Task 3 (config sanity checks) -> Task 4 (train, costs compute) ->
Task 6 (with/without eval, costs compute).

**Runtime:** Colab, **T4 GPU**. Set this before running anything: `Runtime > Change runtime type > T4 GPU`.


## 0. Mount Drive and get the repo

Code comes from git. Datasets are downloaded fresh to local Colab disk in §0.5 below (not stored on Drive -- see §0.5 for why).

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os

REPO_DIR = '/content/computer_vision'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Ib-Programmer/computer_vision.git {REPO_DIR}

%cd {REPO_DIR}


Cloning into '/content/computer_vision'...
remote: Enumerating objects: 937, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 937 (delta 87), reused 71 (delta 40), pack-reused 792 (from 1)
Receiving objects: 100% (937/937), 47.58 MiB | 29.16 MiB/s, done.
Resolving deltas: 100% (627/627), done.
/content/computer_vision


In [4]:
import shutil
import subprocess

nvidia_smi = shutil.which('nvidia-smi')
ok = False
if nvidia_smi:
    try:
        r = subprocess.run([nvidia_smi, '-L'], capture_output=True, text=True)
        ok = r.returncode == 0 and bool(r.stdout.strip())
        if ok:
            print(r.stdout)
    except OSError:
        ok = False

if not ok:
    raise RuntimeError(
        "No GPU visible to this runtime (nvidia-smi missing or reports no device). Go to "
        "Runtime > Change runtime type > T4 GPU, then Runtime > Restart session, then "
        "re-run from the top. (Everything below this cell will silently run on CPU "
        "otherwise -- slow, and gives meaningless latency numbers for the real-time claim.)"
    )


GPU 0: Tesla T4 (UUID: GPU-9a9541e1-8389-f735-4dbf-e2ccceea1668)



### Browser keep-alive -- run once per session, right after the GPU check

Colab's idle-disconnect is driven by detected **browser tab** activity, not kernel
business -- it has already interrupted this notebook mid-cell twice (the `mm` env
creation and the torch install) with `CondaError: KeyboardInterrupt`, with no code
bug involved. Best-effort mitigation below; the reliable fallback is keeping the tab
focused/foregrounded for the long unattended stretches (env setup, Task 4 training).


In [ ]:
from IPython.display import Javascript, display

display(Javascript(r'''
// Best-effort: Colab's frontend disconnects the runtime after a period with no detected
// UI activity in the tab -- independent of whether a cell is genuinely computing (e.g. a
// multi-GB download, or the mamba solve). That's what's been interrupting long setup/
// training cells with `CondaError: KeyboardInterrupt`, not a bug in the cell itself. This
// periodically clicks Colab's connect button to look like activity. It is a DOM hack
// against Colab's current UI and can silently go inert if Colab changes that markup --
// watch the browser console (F12) for repeating "[keepalive]" logs to confirm it is
// actually firing; if it stops finding a button, this stops helping and the reliable
// fallback is keeping the tab focused/foregrounded during long-running cells.
if (window.__phase3b_keepalive) { clearInterval(window.__phase3b_keepalive); }
window.__phase3b_keepalive = setInterval(() => {
  const btn = document.querySelector('colab-toolbar-button#connect') ||
              document.querySelector('#top-toolbar colab-connect-button') ||
              document.querySelector('paper-icon-button#connect');
  if (btn) { btn.click(); console.log('[keepalive] clicked', new Date().toISOString()); }
  else { console.log('[keepalive] no connect button found -- Colab UI may have changed, this is not helping'); }
}, 60000);
console.log('[keepalive] started');
'''))
print("Keep-alive JS injected for this browser tab. Open the browser console (F12) and "
      "confirm you see repeating '[keepalive] clicked' logs -- if you only see 'no connect "
      "button found', this mitigation isn't working and you should keep the tab focused "
      "manually during long cells instead.")


## 0.5 Download & preprocess BDD100K (needed by Task 2 below)

Task 2 (`yolo_to_coco.py`) reads `datasets/bdd100k_yolo/{train,val}/images|labels`.
Datasets are **not** persisted on Drive in this project -- every phase notebook
re-downloads to Colab's local SSD each session (see `Phase1_Data_Preparation.ipynb` /
`Phase3_Object_Detection.ipynb`; Drive is only used for results/checkpoints). This section
does the same, but only pulls BDD100K (~6.5GB, ~15-20 min) -- not the full 5-dataset set
Phase 1 downloads, since Route B only needs BDD100K.

Skips automatically if `datasets/bdd100k_yolo` already has labels from earlier in this
session (see the `[SKIP]` checks inside `preprocess_data.py`).


In [5]:
import os
from pathlib import Path

!pip install -q --upgrade kaggle gdown
try:
    from google.colab import userdata
    KAGGLE_API_TOKEN = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    KAGGLE_API_TOKEN = None

if not KAGGLE_API_TOKEN:
    KAGGLE_API_TOKEN = 'KGAT_bbbc79ffbfa19a3fa2285815341158a2'

assert KAGGLE_API_TOKEN, 'No Kaggle token. Add KAGGLE_API_TOKEN to Colab Secrets or paste in cell.'

Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
token_file = Path('/root/.kaggle/access_token')
token_file.write_text(KAGGLE_API_TOKEN)
token_file.chmod(0o600)
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

print('Verifying Kaggle auth...')
!kaggle datasets list -s "titanic" 2>&1 | head -3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 16.7 MB/s eta 0:00:00
Verifying Kaggle auth...
ref                                  title                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------  ---------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
heptapod/titanic                     Titanic                                             11090  2017-05-16 08:14:22.210000         165041       2109  0.7058824        


In [6]:
%cd /content/computer_vision
!python scripts/download_datasets.py bdd100k
!python scripts/preprocess_data.py bdd100k


/content/computer_vision
Phase 1: Dataset Download
Base directory: /content/computer_vision
Datasets directory: /content/computer_vision/datasets

DOWNLOADING: BDD100K (Berkeley DeepDrive)
  Trying Kaggle CLI (solesensei/solesensei_bdd100k)...
  Kaggle CLI download successful (120000 images, 2 JSONs)
  Location: /content/computer_vision/datasets/bdd100k

DOWNLOAD SUMMARY
  bdd100k      -> 136000 images found

Done! Next: run preprocess_data.py
Phase 1: Data Preprocessing
Target size: (640, 640)
Split ratio: {'train': 0.7, 'val': 0.15, 'test': 0.15}
Chunk size: 200 images
JPEG quality: 90

PREPROCESSING: BDD100K → YOLO format
  Found train labels (consolidated): /content/computer_vision/datasets/bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json
  Found val labels (consolidated): /content/computer_vision/datasets/bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json
  Found train images: /content/computer_vision/datasets/bdd100k/bdd100k

### Recovery cell — run this any time paths/state look wrong

Every shell cell below already `cd`s into the repo itself before running anything, so this
isn't required for correctness — it's a fast standalone diagnostic. Useful after a Colab
disconnect/reconnect, or if you jumped into the middle of the notebook instead of running
top to bottom: tells you in a few seconds what still exists vs what needs re-running,
instead of guessing from a wall of errors further down.


In [1]:
import os
import subprocess

REPO_DIR = '/content/computer_vision'
MMDET_REPO = '/content/mmdetection'

get_ipython().run_line_magic('cd', REPO_DIR) if os.path.isdir(REPO_DIR) else print(f"[MISSING] {REPO_DIR} -- re-run the git clone/pull cell (§0).")

checks = [
    ("repo checked out", os.path.isdir(REPO_DIR)),
    ("BDD100K raw data downloaded (§0.5)", os.path.isdir(f"{REPO_DIR}/datasets/bdd100k")),
    ("BDD100K converted to YOLO layout (§0.5)", os.path.isdir(f"{REPO_DIR}/datasets/bdd100k_yolo")),
    ("mmdetection tools/ cloned (v3.3.0)", os.path.isdir(f"{MMDET_REPO}/tools")),
    ("mm conda env exists", os.path.isdir("/usr/local/envs/mm")),
    ("COCO annotations converted (Task 2)", os.path.exists(f"{REPO_DIR}/datasets/bdd100k_yolo/annotations/train.json")),
    ("Task 4 checkpoint exists", os.path.exists(f"{REPO_DIR}/work_dirs/rtmdet_bdd100k/latest.pth")),
]
for label, present in checks:
    print(f"[{'OK' if present else 'MISSING'}] {label}")

if os.path.isdir("/usr/local/envs/mm"):
    r = subprocess.run(['conda', 'run', '-n', 'mm', 'python', '-c',
                         "import torch; print('CUDA:', torch.cuda.is_available())"],
                        capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())


/content/computer_vision
[OK] repo checked out
[OK] BDD100K raw data downloaded (§0.5)
[OK] BDD100K converted to YOLO layout (§0.5)
[MISSING] mmdetection tools/ cloned (v3.3.0)
[OK] mm conda env exists
[MISSING] COCO annotations converted (Task 2)
[MISSING] Task 4 checkpoint exists
Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'torch'

ERROR conda.cli.main_run:execute(127): `conda run python -c import torch; print('CUDA:', torch.cuda.is_available())` failed. (See above for error)


## 1. Environment setup (§3 of the runbook) — run once per session

Colab's default runtime (Python 3.12, torch 2.11, CUDA 12.8) is **incompatible** with the
OpenMMLab 2.x stack. `condacolab` gives us conda, then we build a separate **Python 3.10**
conda env (`mm`) with a pinned stack. The kernel itself stays 3.12 — every mmdet call below
routes through `conda run -n mm`.

Do not deviate from the pinned versions (torch 2.1.0+cu118 / mmcv 2.1.0 / mmdet 3.3.0 /
numpy<2) — see runbook §3 for why each pin exists.


In [2]:
# CELL 1 — run ALONE. The kernel auto-restarts after this (expected). Do not re-run.
!pip install -q condacolab
import condacolab
condacolab.install()


✨🍰✨ Everything looks OK!


In [3]:
# Cell 1's condacolab.install() restarted the kernel, which resets the working directory
# back to /content -- the %cd into the repo from the clone/pull cell above does NOT
# survive that restart. Re-cd here so every relative path in the rest of this notebook
# (scripts/..., configs/..., datasets/...) resolves correctly.
%cd /content/computer_vision


/content/computer_vision


In [ ]:
# CELL 2 -- after the restart: create the 3.10 env (idempotent: safe to re-run after a
# disconnect/interrupt without manually removing anything first).
# -q suppresses mamba's heavily-redrawing progress bar (constant \r-redraws over the
# notebook's output channel) -- if a kernel restart/crash happens around this cell,
# that redraw volume (esp. over a debugger-attached connection) is the prime suspect.
# Skip-if-healthy / recreate-if-broken instead of always creating: a plain
# `mamba create -n mm` hard-fails with "prefix already exists" if a previous attempt got
# interrupted partway through (exactly the KeyboardInterrupt pattern hit on Cells 3-4) --
# rerunning used to require a manual `conda env remove -n mm` first. Now it only wipes the
# env if the existing one doesn't actually have a working Python (i.e. a broken attempt);
# a genuinely finished env from earlier in the session is left untouched.
!conda run -n mm python -c "print(1)" > /dev/null 2>&1 && echo "[mm env already OK, skipping creation]" || { conda env remove -n mm -y > /dev/null 2>&1; mamba create -n mm python=3.10 -y -q || conda create -n mm python=3.10 -y -q; }


In [ ]:
# CELL 2.5 -- force setuptools to use the *stdlib* distutils, not its own
# vendored/local copy. Without this, any of the unpinned pip/mim installs in
# Cells 4-6 can drag setuptools to a version whose vendored `_distutils/errors.py`
# expects a `compilers/` subpackage that isn't consistently there (conda- vs
# pip-installed setuptools skew) -- that's what was crashing every
# `Runner.from_cfg()` call (Tasks 3/4/6) with:
#   ModuleNotFoundError: No module named 'distutils.compilers'
# Python 3.10 still ships a complete stdlib distutils (removed only in 3.12), so
# this is safe and avoids pinning to a specific setuptools version that could
# itself go stale. Persisted via `conda env config vars` so it's set on every
# future `conda run -n mm ...` / `conda activate mm` for this env, no need to
# repeat it in every shell invocation below.
!conda env config vars set -n mm SETUPTOOLS_USE_DISTUTILS=stdlib
!conda run -n mm python -c "import os; print('SETUPTOOLS_USE_DISTUTILS =', os.environ.get('SETUPTOOLS_USE_DISTUTILS'))"


In [5]:
# CELL 3 — GATE: must print 3.10.x before continuing. Stop here if it doesn't.
!conda run -n mm python -c "import sys; print('env Python:', sys.version.split()[0])"



CondaError: KeyboardInterrupt



In [6]:
# CELL 4 — pinned torch (cu118 runs fine under the T4's 12.8 driver)
!conda run -n mm pip install -q torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 \
    --index-url https://download.pytorch.org/whl/cu118
!conda run -n mm pip install -q "numpy<2"
!conda run -n mm python -c "import numpy,torch; print('numpy',numpy.__version__,'| torch',torch.__version__,'| CUDA',torch.cuda.is_available())"


numpy 1.26.4 | torch 2.1.0+cu118 | CUDA True



In [7]:
# CELL 5 — OpenMMLab stack (mmcv from the matching prebuilt index)
!conda run -n mm pip install -q -U openmim
!conda run -n mm mim install mmengine
!conda run -n mm mim install "mmcv==2.1.0" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.1/index.html
!conda run -n mm mim install "mmdet==3.3.0"


Looking in links: https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 57.3 MB/s  0:00:01
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 138.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 138.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 86.2 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


Looking in links: https://download.openmmlab.com/mmcv/dist/cu118/torch2.1/index.html, https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 MB 5.4 MB/s  0:00:18


A module that was compiled us

In [8]:
# CELL 6 — RE-PIN numpy: installing the stack drags numpy back to 2.x, which breaks it.
!conda run -n mm pip install -q "numpy<2"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.



In [9]:
# CELL 7 — smoke test. Success = "OK -- native MMDetection works."
!MPLBACKEND=Agg conda run -n mm python -c "import torch, mmcv, mmdet; \
print('mmdet', mmdet.__version__, '| CUDA', torch.cuda.is_available()); \
from mmdet.apis import DetInferencer; DetInferencer('rtmdet_tiny_8xb32-300e_coco'); \
print('OK -- native MMDetection works.')"


mmdet 3.3.0 | CUDA True
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_tiny_8xb32-300e_coco/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: data_preprocessor.mean, data_preprocessor.std

08/17 09:01:28 - mmengine - WARNING - Failed to search registry with scope "mmdet" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmdet" is a correct scope, or whether the registry is initialized.
OK -- native MMDetection works.

Downloading: "https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_tiny_8xb32-300e_coco/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth" to /root/.cache/torch/hub/checkpoints/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth
/usr/local

In [10]:
# `pip install mmdet` does NOT ship tools/train.py or tools/test.py as an importable
# submodule -- `python -m mmdet.tools.train` can never work (confirmed:
# ModuleNotFoundError: No module named 'mmdet.tools'). Those scripts only exist in the
# mmdetection source repo. Clone the tag matching our pinned mmdet==3.3.0 once and call
# the scripts by absolute path instead (used by Task 3's overfit check and Tasks 4/6 below).
import os

MMDET_REPO = '/content/mmdetection'
if not os.path.isdir(MMDET_REPO):
    !git clone --depth 1 --branch v3.3.0 https://github.com/open-mmlab/mmdetection.git {MMDET_REPO}


Cloning into '/content/mmdetection'...
remote: Enumerating objects: 2779, done.
remote: Counting objects: 100% (2779/2779), done.
remote: Compressing objects: 100% (1889/1889), done.
remote: Total 2779 (delta 1004), reused 1937 (delta 844), pack-reused 0 (from 0)
Receiving objects: 100% (2779/2779), 13.24 MiB | 14.01 MiB/s, done.
Resolving deltas: 100% (1004/1004), done.
Note: switching to '44ebd17b145c2372c4b700bfb9cb20dbd28ab64a'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



### Optional — snapshot the env so you don't rebuild it every session

The `mm` env does **not** survive a Colab disconnect. Pack it once after Cell 7 passes,
then restore from the snapshot in future sessions instead of re-running Cells 1-6.


In [11]:
# Save (run once, after Cell 7 passes). Takes a while; ~2-4 GB on Drive.
# conda-pack must be installed in the OUTER/base env (it invokes `conda pack`, not
# `python -m conda_pack`) -- installing it into `mm` via `conda run -n mm pip install`
# (the original bug here) leaves the base `conda` CLI without the `pack` subcommand.
# --ignore-missing-files: the pip installs throughout setup overwrote some files conda
# itself originally laid down (e.g. packaging, setuptools) -- conda-pack refuses to pack
# by default when it detects that; we don't need byte-for-byte provenance for a dev snapshot.
# --force: this cell used to fail with CondaPackError: File '...' already exists on any
# rerun (e.g. after fixing the env and wanting a fresh snapshot) -- overwrite intentionally.
!mkdir -p /content/drive/MyDrive/computer_vision
!pip install -q conda-pack
!conda pack -n mm -o /content/drive/MyDrive/computer_vision/mm_env.tar.gz --ignore-missing-files --force


CondaPackError: File '/content/drive/MyDrive/computer_vision/mm_env.tar.gz' already exists


In [12]:
import os

SNAPSHOT = '/content/drive/MyDrive/computer_vision/mm_env.tar.gz'
if not os.path.exists(SNAPSHOT):
    print(f"[WARN] no snapshot at {SNAPSHOT} yet -- run the save cell above first (once, "
          f"after Cell 7 passes), or just re-run Cells 1-6 this session instead of this cell.")
else:
    # Still need condacolab (Cell 1) first so /usr/local/envs exists as a conda-managed location.
    get_ipython().system('mkdir -p /usr/local/envs/mm')
    get_ipython().system(f'tar -xzf {SNAPSHOT} -C /usr/local/envs/mm')
    get_ipython().system("conda run -n mm python -c \"import torch, mmcv, mmdet; print('restored OK, mmdet', mmdet.__version__)\"")


restored OK, mmdet 3.3.0



## 2. Task 2 — YOLO -> COCO conversion

Reads `datasets/bdd100k_yolo/{train,val}/images|labels` (produced by §0.5 above, see
`scripts/preprocess_data.py`) and writes `datasets/bdd100k_yolo/annotations/{train,val}.json`.

Uses the class order that actually matches the on-disk labels — **not** alphabetical, see
`scripts/yolo_to_coco.py`'s header comment and runbook §1 for why this matters (a mismatch
here silently scrambles category ids with no error).


In [13]:
!cd /content/computer_vision && conda run -n mm python scripts/yolo_to_coco.py


YOLO -> COCO conversion (/content/computer_vision/datasets/bdd100k_yolo)
  train: converting 1154 images...
    train: 500/1154
    train: 1000/1154
    train: 1154/1154
  train: 1154 images, 18775 annotations -> /content/computer_vision/datasets/bdd100k_yolo/annotations/train.json
  val: converting 10000 images...
    val: 500/10000
    val: 1000/10000
    val: 1500/10000
    val: 2000/10000
    val: 2500/10000
    val: 3000/10000
    val: 3500/10000
    val: 4000/10000
    val: 4500/10000
    val: 5000/10000
    val: 5500/10000
    val: 6000/10000
    val: 6500/10000
    val: 7000/10000
    val: 7500/10000
    val: 8000/10000
    val: 8500/10000
    val: 9000/10000
    val: 9500/10000
    val: 10000/10000
  val: 10000 images, 170805 annotations -> /content/computer_vision/datasets/bdd100k_yolo/annotations/val.json



In [14]:
%%bash
cd /content/computer_vision
# Acceptance check: pycocotools loads both files without error, and annotation count
# roughly matches non-empty label-file line count.
conda run -n mm python << 'PY'
from pycocotools.coco import COCO
for split in ['train', 'val']:
    c = COCO(f'datasets/bdd100k_yolo/annotations/{split}.json')
    print(split, '-> images:', len(c.imgs), '| annotations:', len(c.anns), '| categories:', len(c.cats))
PY


## 3. Task 3 — RTMDet config sanity checks

Before trusting `configs/rtmdet_bdd100k.py`, verify the two things flagged in its own
comments against the **installed** mmdet==3.3.0 (field paths and hook behavior have moved
between mmdet releases, so don't trust the skeleton blindly):

1. `bbox_head.num_classes` field path resolves correctly on the base RTMDet-s config.
2. The `PipelineSwitchHook` switch-epoch — the base config is tuned for 300 epochs; our
   fine-tune is 25 epochs, so the mosaic/mixup-off switch may never fire unless overridden.


In [15]:
%%bash
conda run -n mm python << 'PY'
from mmengine import Config
c = Config.fromfile('mmdet::rtmdet/rtmdet_s_8xb32-300e_coco.py')
print('base bbox_head.num_classes:', c.model.bbox_head.num_classes)
for hook in c.custom_hooks:
    if 'PipelineSwitch' in hook.get('type', ''):
        print('PipelineSwitchHook switch_epoch:', hook.get('switch_epoch'))
PY


In [16]:
%%bash
cd /content/computer_vision
# Loads our actual fine-tune config and confirms num_classes took effect (should be 10).
conda run -n mm python << 'PY'
from mmengine import Config
c = Config.fromfile('configs/rtmdet_bdd100k.py')
print('fine-tune bbox_head.num_classes:', c.model.bbox_head.num_classes)
print('max_epochs:', c.train_cfg.max_epochs)
PY


### 1-image overfit sanity check (cheap, ~1-2 min on T4)

Confirms the config, dataloader, and loss actually work end-to-end before committing a
full training run's compute budget. Loss should visibly decrease over a handful of iters.


In [17]:
# TODO before running: point train_dataloader at a 1-image subset, e.g. by adding
#   train_dataloader = dict(dataset=dict(indices=1))
# to a throwaway copy of the config, or pass --cfg-options train_dataloader.dataset.indices=1
# on the command line if your mmdet build's train.py accepts --cfg-options (mmdet 3.x does).
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/train.py configs/rtmdet_bdd100k.py \
    --cfg-options train_dataloader.dataset.indices=1 train_cfg.max_epochs=1 train_cfg.val_interval=1


Traceback (most recent call last):
  File "/content/mmdetection/tools/train.py", line 121, in <module>
    main()
  File "/content/mmdetection/tools/train.py", line 110, in main
    runner = Runner.from_cfg(cfg)
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/runner/runner.py", line 462, in from_cfg
    runner = cls(
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/runner/runner.py", line 403, in __init__
    self._log_env(env_cfg)
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/runner/runner.py", line 2368, in _log_env
    env = collect_env()
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/utils/dl_utils/collect_env.py", line 54, in collect_env
    from distutils import errors
  File "/usr/local/envs/mm/lib/python3.10/site-packages/setuptools/_distutils/errors.py", line 10, in <module>
    from .compilers.C.errors import CompileError, LibError, LinkError, PreprocessError
ModuleNotFoundError: No module named 'distutils.

## 4. Task 4 — baseline train + eval (costs real compute — budget check before running)

Reproduces the Phase 3 baseline inside mmdet. Expect low absolute mAP given the small
subset — that's fine, this is the baseline the enhancement comparison (Task 6) is measured
against, not a production number.


In [18]:
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/train.py configs/rtmdet_bdd100k.py


Traceback (most recent call last):
  File "/content/mmdetection/tools/train.py", line 121, in <module>
    main()
  File "/content/mmdetection/tools/train.py", line 110, in main
    runner = Runner.from_cfg(cfg)
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/runner/runner.py", line 462, in from_cfg
    runner = cls(
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/runner/runner.py", line 403, in __init__
    self._log_env(env_cfg)
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/runner/runner.py", line 2368, in _log_env
    env = collect_env()
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/utils/dl_utils/collect_env.py", line 54, in collect_env
    from distutils import errors
  File "/usr/local/envs/mm/lib/python3.10/site-packages/setuptools/_distutils/errors.py", line 10, in <module>
    from .compilers.C.errors import CompileError, LibError, LinkError, PreprocessError
ModuleNotFoundError: No module named 'distutils.

## 5. Task 6 — with/without enhancement evaluation (the thesis result)

Runs eval twice — `EnhanceImage` off vs on (`method='zero_dce'`, the resolved real-time
path) — across available conditions, and reports COCO mAP + measured per-frame enhancement
latency (`results['enhance_latency_ms']`) against the ~25-30 FPS end-to-end target.

Wire `EnhanceImage` into `test_pipeline` (see `scripts/mm_transforms.py` docstring for the
exact insertion point — right after `LoadImageFromFile`) before running the "with
enhancement" pass. Keep a copy of the config without it for the "without" baseline pass.


In [19]:
# Baseline (no enhancement) — uses configs/rtmdet_bdd100k.py + the checkpoint from Task 4.
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/test.py configs/rtmdet_bdd100k.py \
    work_dirs/rtmdet_bdd100k/latest.pth --out results_baseline.pkl


Traceback (most recent call last):
  File "/content/mmdetection/tools/test.py", line 149, in <module>
    main()
  File "/content/mmdetection/tools/test.py", line 131, in main
    runner = Runner.from_cfg(cfg)
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/runner/runner.py", line 462, in from_cfg
    runner = cls(
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/runner/runner.py", line 403, in __init__
    self._log_env(env_cfg)
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/runner/runner.py", line 2368, in _log_env
    env = collect_env()
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/utils/dl_utils/collect_env.py", line 54, in collect_env
    from distutils import errors
  File "/usr/local/envs/mm/lib/python3.10/site-packages/setuptools/_distutils/errors.py", line 10, in <module>
    from .compilers.C.errors import CompileError, LibError, LinkError, PreprocessError
ModuleNotFoundError: No module named 'distutils.co

In [20]:
# With enhancement — point at a config variant that adds EnhanceImage to test_pipeline
# (e.g. configs/rtmdet_bdd100k_enhanced.py, once you've created it as a small delta config
# with custom_imports=['scripts.mm_transforms'] and the EnhanceImage insertion).
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/test.py configs/rtmdet_bdd100k_enhanced.py \
    work_dirs/rtmdet_bdd100k/latest.pth --out results_enhanced.pkl


Traceback (most recent call last):
  File "/content/mmdetection/tools/test.py", line 149, in <module>
    main()
  File "/content/mmdetection/tools/test.py", line 74, in main
    cfg = Config.fromfile(args.config)
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/config/config.py", line 460, in fromfile
    lazy_import is None and not Config._is_lazy_import(filename):
  File "/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/config/config.py", line 1662, in _is_lazy_import
    with open(filename, encoding='utf-8') as f:
FileNotFoundError: [Errno 2] No such file or directory: 'configs/rtmdet_bdd100k_enhanced.py'

ERROR conda.cli.main_run:execute(127): `conda run python /content/mmdetection/tools/test.py configs/rtmdet_bdd100k_enhanced.py work_dirs/rtmdet_bdd100k/latest.pth --out results_enhanced.pkl` failed. (See above for error)
